# Local host-hour reconstruction study

One bounded comparison of robust scores, PCA and a small autoencoder. A positive result concerns incremental ranking of the named injected cases under frozen thresholds, not general attack discrimination.

The default route below is entirely synthetic and requires no private inputs. The optional real-study route uses a separately supplied local runner and keeps source identities, operational configuration, raw rows, frozen fitted objects and transformed copies outside both repositories. Real outputs are cleared from this notebook.

The study uses a 28-day reference, seven-day evaluation weeks, 32 features in absolute and host-relative blocks, and explicit abstention for insufficient history. Read the definition cells for every eligibility, scaling, folding and attribution rule.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "sigwood").is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

In [2]:
"""Definitions embedded verbatim in conn_hosthour.ipynb; never a product module."""
from __future__ import annotations
import contextlib, collections, gc, hashlib, importlib.metadata, ipaddress, json, math, os, pickle, platform, re, resource, signal, subprocess, sys, tempfile, time, warnings
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning
from sigwood.common import loader
from sigwood.detectors.beacon import _is_non_unicast, _ip_in_home_net, _DEFAULT_HOME_NET
from sigwood.parsers.zeek import parse_service

SEEDS = [20261025, 20261026, 20261027, 20261028, 20261029]
BASE_NAMES = ['conns','orig_bytes','resp_bytes','orig_share','byte_coverage','internal_dst','external_dst','dst_ports','port_missing_share','external_share','tcp_share','udp_share','icmp_share']
TAIL_NAMES = ['service_entropy','non_established_share','rej_share','s0_share','state_missing_share','dur_median','dur_p95','duration_coverage','active_minutes']
METHODS = ['robust-z','PCA','autoencoder']
PARAMETERS = dict(hour_floor=5, history_days=7, history_hours=100, trim=10, threshold_percentile=99.5, quantile_method='linear', K=10, seeds=SEEDS, topology=[32,8,32], max_iter=500, batch_size=256, validation_fraction=.1, n_iter_no_change=10, tol=1e-4, service_slots=8, scale_range_floor=.05, gap_hours=1)
REFERENCE = [date(2026,6,15)+timedelta(days=i) for i in range(28)]
WEEKS = [[date(2026,8,3)+timedelta(days=i) for i in range(7)], [date(2026,8,17)+timedelta(days=i) for i in range(7)]]
ALLOWED = set(REFERENCE + WEEKS[0] + WEEKS[1])
RESERVE = {date(2026,8,24)+timedelta(days=i) for i in range(14)}


def canonical(x):
    return json.dumps(x,sort_keys=True,separators=(',',':'),allow_nan=False,default=lambda v: v.item() if isinstance(v,np.generic) else (_ for _ in ()).throw(TypeError(type(v).__name__))).encode()

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(2**20),b''): h.update(block)
    return h.hexdigest()

def json_write(path,value):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_suffix(path.suffix+'.tmp'); temp.write_bytes(canonical(value)); temp.replace(path)

def instant(): return datetime.now(timezone.utc).isoformat()

def guard_day(day):
    day=date.fromisoformat(str(day))
    assert day not in RESERVE, 'reserve_refused'
    assert day in ALLOWED, 'date_outside_preregistration'
    return day

def parity_checks():
    assert not _is_non_unicast('192.0.2.1') and _is_non_unicast('169.254.1.1')
    assert _is_non_unicast('224.0.0.1') and _is_non_unicast('255.255.255.255')
    assert _ip_in_home_net('192.0.2.1',['192.0.2.0/24'])
    assert not _ip_in_home_net('198.51.100.1',['192.0.2.0/24'])
    try: guard_day('2026-08-24')
    except AssertionError as e: assert str(e)=='reserve_refused'
    else: raise AssertionError('reserve guard failed')
    assert parse_service('ssl,http,-dns') == (('http','ssl'),('dns',))
    return {'address_parity':'passed','reserve_guard_self_test':'passed; no file discovery/open','features':32,'matrix_columns_before_constants':64}


def valid_address(x):
    try: ipaddress.ip_address(x); return True
    except (ValueError,TypeError): return False


def aggregate(frame, networks, coverage, period):
    """Keep only compact host/hour features; no raw rows escape the caller."""
    df=frame.copy(deep=False)
    counts=collections.Counter(input_rows=len(df))
    if df.empty: coverage.append({'period':period,**counts}); return []
    ts=pd.to_numeric(df['ts'],errors='coerce'); mask=np.isfinite(ts)
    counts['invalid_ts']=int((~mask).sum()); df=df.loc[mask].copy(); df['ts']=ts.loc[mask]
    for col in ('src','dst'):
        unique=df[col].drop_duplicates(); good={v:valid_address(v) for v in unique}
        keep=df[col].map(good).fillna(False).astype(bool)
        counts['invalid_'+col]=int((~keep).sum()); df=df.loc[keep]
        keep=~df[col].map({v:_is_non_unicast(v) for v in df[col].unique()}).astype(bool)
        counts['non_unicast_'+col]=int((~keep).sum()); df=df.loc[keep]
    local=df['src'].map({v:_ip_in_home_net(v,networks) for v in df['src'].unique()})
    lo=df.get('local_orig',pd.Series(pd.NA,index=df.index))
    keep=lo.where(lo.notna(),local).astype(bool)
    counts['not_local']=int((~keep).sum()); df=df.loc[keep].copy()
    counts['eligible_rows']=len(df)
    if df.empty: coverage.append({'period':period,**counts}); return []
    df['_hour']=(df['ts'].to_numpy()//3600).astype('int64')*3600
    df['_internal']=df['dst'].map({v:_ip_in_home_net(v,networks) for v in df['dst'].unique()})
    services=df.get('service',pd.Series(None,index=df.index)).fillna('')
    cats={v:('+'.join(parse_service(v)[0]) or 'none') for v in services.unique()}
    df['_cat']=services.map(cats)
    for col in ('bytes','resp_bytes','duration','port'):
        if col not in df: df[col]=np.nan
        df[col]=pd.to_numeric(df[col],errors='coerce')
    if 'conn_state' not in df: df['conn_state']=None
    if 'proto' not in df: df['proto']=None
    rows=[]
    for (host,hour),g in df.groupby(['src','_hour'],sort=True):
        n=len(g); day=datetime.fromtimestamp(int(hour),timezone.utc).date().isoformat()
        base={'period':period,'host':host,'day':day,'hour':int(hour),'rows':n}
        if n<5: coverage.append({**base,'reason':'below_floor'}); continue
        b=g[['bytes','resp_bytes']].to_numpy(float); pairs=np.isfinite(b).all(axis=1)&(b>=0).all(axis=1)
        if not pairs.any(): coverage.append({**base,'reason':'no_byte_pair'}); continue
        sums=b[pairs].sum(axis=0); total=float(sums.sum()); internal=g['_internal'].astype(bool)
        counts_cat=collections.Counter(g['_cat']); probs=np.array(list(counts_cat.values()),float)/n
        entropy=float(-(probs*np.log(probs)).sum()/np.log(len(probs))) if len(probs)>1 else 0.
        states=g['conn_state'].dropna(); ns=len(states); dur=g['duration'].to_numpy(float); dur=dur[np.isfinite(dur)&(dur>=0)]
        proto=g['proto'].fillna('').astype(str).str.lower()
        x=np.zeros(32,float)
        x[:13]=[np.log1p(n),np.log1p(sums[0]),np.log1p(sums[1]),sums[0]/total if total else .5,float(pairs.mean()),np.log1p(g.loc[internal,'dst'].nunique()),np.log1p(g.loc[~internal,'dst'].nunique()),np.log1p(g['port'].nunique()),float(g['port'].isna().mean()),float((~internal).mean()),float((proto=='tcp').mean()),float((proto=='udp').mean()),float((proto=='icmp').mean())]
        x[23:]=[entropy,float((~states.isin(['SF','S1'])).mean()) if ns else 0.,float((states=='REJ').mean()) if ns else 0.,float((states=='S0').mean()) if ns else 0.,1-ns/n,np.log1p(np.median(dur)) if len(dur) else 0.,np.log1p(np.percentile(dur,95,method='linear')) if len(dur) else 0.,len(dur)/n,g['ts'].floordiv(60).nunique()/60]
        assert np.isfinite(x).all() and len(x)==32
        rows.append({'host':host,'hour':int(hour),'day':day,'x':x,'cats':dict(counts_cat),'n':n})
        coverage.append({**base,'reason':'admitted'})
    coverage.append({'period':period,**counts})
    return rows


def scale_fit(x):
    med=np.median(x,axis=0); iqr=np.percentile(x,75,axis=0)-np.percentile(x,25,axis=0)
    mad=1.4826*np.median(np.abs(x-med),axis=0); spread=np.ptp(x,axis=0)
    divisor=np.where(iqr>0,iqr,np.where(mad>0,mad,spread))
    divisor=np.maximum(divisor,.05*spread); keep=spread>0
    divisor[~keep]=1
    return {'center':med,'divisor':divisor,'keep':keep}

def scale_apply(x,s): return (x-s['center'])/s['divisor']

def set_services(rows,vocab):
    for row in rows:
        x=row['x']; cats=row['cats']; n=row['n']; x[13:23]=0
        for i,cat in enumerate(vocab): x[13+i]=cats.get(cat,0)/n
        x[21]=sum(v for c,v in cats.items() if c!='none' and c not in vocab)/n
        x[22]=cats.get('none',0)/n
        assert np.isclose(x[13:23].sum(),1)


def named_features(vocab):
    names=BASE_NAMES+[('svc_'+vocab[i]) if i<len(vocab) else 'svc_pad_'+str(i+1) for i in range(8)]+['svc_other','svc_none']+TAIL_NAMES
    assert len(names)==32
    return names


def training(reference,networks):
    subset=list(range(13))+list(range(23,32)); pre=np.stack([r['x'] for r in reference]); trim_scaler=scale_fit(pre[:,subset])
    score=np.max(np.abs(scale_apply(pre[:,subset],trim_scaler)[:,trim_scaler['keep']]),axis=1) if trim_scaler['keep'].any() else np.zeros(len(pre))
    ref=[r for r,s in zip(reference,score,strict=True) if s<=10]
    assert ref,'empty post-trim reference'
    cats=collections.Counter()
    for r in ref: cats.update({c:v for c,v in r['cats'].items() if c!='none'})
    ranking=sorted(cats,key=lambda c:(-cats[c],c)); vocab=ranking[:8]; set_services(ref,vocab)
    groups=collections.defaultdict(list)
    for r in ref: groups[r['host']].append(r)
    medians={}; history=[]
    for host,rows in sorted(groups.items()):
        days=len({r['day'] for r in rows}); ok=len(rows)>=100 and days>=7
        history.append({'host':host,'hours':len(rows),'days':days,'scorable':ok})
        if ok: medians[host]=np.median(np.stack([r['x'] for r in rows]),axis=0)
    ref=[r for r in ref if r['host'] in medians]; assert ref,'no scorable reference hosts'
    names=named_features(vocab); full=np.stack([np.r_[r['x'],r['x']-medians[r['host']]] for r in ref])
    scaler=scale_fit(full); scaled=scale_apply(full,scaler); keep=scaler['keep']; assert keep.any(),'all columns constant'
    colnames=['A.'+n for n in names]+['B.'+n for n in names]
    # Stable content order, with canonical host and hour only as tie-breakers.
    order=sorted(range(len(ref)),key=lambda i:(tuple(scaled[i,keep]),ref[i]['host'],ref[i]['hour']))
    scaled=scaled[order]; ref=[ref[i] for i in order]
    models={}; thresholds={}; fits=[]; fit_times={}
    for block,mask in [('full',keep),('A',keep & (np.arange(64)<32)),('B',keep & (np.arange(64)>=32))]:
        assert mask.any(),'empty block '+block
        x=scaled[:,mask]; block_models={}; block_thresholds={}
        for method in METHODS:
            seeds=SEEDS if method=='autoencoder' else [SEEDS[0]]
            for seed in seeds:
                start=time.perf_counter(); caught=[]; model=None
                if method=='PCA': model=PCA(n_components=.90,svd_solver='full',whiten=False,random_state=seed); model.fit(x)
                if method=='autoencoder':
                    model=MLPRegressor(hidden_layer_sizes=(32,8,32),activation='relu',solver='adam',alpha=1e-4,learning_rate_init=1e-3,batch_size=256,max_iter=500,early_stopping=True,validation_fraction=.1,n_iter_no_change=10,tol=1e-4,shuffle=True,random_state=seed)
                    with warnings.catch_warnings(record=True) as caught:
                        warnings.simplefilter('always'); model.fit(x,x)
                scores,parts=model_scores(method,model,x)
                assert np.isfinite(scores).all()
                key=method if seed==SEEDS[0] else method+'@'+str(seed)
                block_models[key]=model; block_thresholds[key]=float(np.percentile(scores,99.5,method='linear'))
                fits.append({'block':block,'method':method,'seed':seed,'seconds':time.perf_counter()-start,'iterations':getattr(model,'n_iter_',None),'convergence_warnings':sum(issubclass(w.category,ConvergenceWarning) for w in caught),'warnings':len(caught)})
        models[block]={'mask':mask,'models':block_models}; thresholds[block]=block_thresholds
    host_order=sorted(medians,key=lambda h:(len(groups[h]),h)); targets=[h for h in host_order if _ip_in_home_net(h,networks)]
    assert len(targets)>=2,'fewer than two in-network scorable targets'
    quiet,busy=targets[0],targets[-1]; remaining=[h for h in targets if h not in (quiet,busy)]
    donor=remaining[(len(remaining)-1)//2] if len(targets)>=3 else None
    mapping={h:'host-%02d'%(i+1) for i,h in enumerate(sorted(groups))}; mapping[quiet]='host-quiet'; mapping[busy]='host-busy'
    if donor: mapping[donor]='host-donor'
    return dict(vocab=vocab,ranking=[{'category':c,'rows':cats[c]} for c in ranking],medians=medians,scaler=scaler,names=names,colnames=colnames,models=models,thresholds=thresholds,fits=fits,history=history,trimmed=len(reference)-sum(s<=10 for s in score),reference_hours=len(ref),targets={'quiet':quiet,'busy':busy},donor=donor,mapping=mapping,networks=networks)


def model_scores(method,model,x):
    if method=='robust-z': parts=np.abs(x); return parts.max(axis=1),parts
    reconstructed=model.inverse_transform(model.transform(x)) if method=='PCA' else model.predict(x)
    parts=np.square(x-reconstructed); return parts.mean(axis=1),parts


def fold_leads(rows,scores,threshold):
    flagged=sorted([(r['host'],r['hour'],float(s),i) for i,(r,s) in enumerate(zip(rows,scores,strict=True)) if s>threshold])
    leads=[]
    for host,hour,score,i in flagged:
        if not leads or host!=leads[-1]['host'] or hour-leads[-1]['hours'][-1]>7200:
            leads.append({'host':host,'hours':[],'indices':[],'score':score})
        lead=leads[-1]; lead['hours'].append(hour); lead['indices'].append(i); lead['score']=max(lead['score'],score)
    leads.sort(key=lambda l:(-l['score'],l['hours'][0],l['host']))
    for rank,lead in enumerate(leads,1): lead['rank']=rank
    return leads


def evaluate(rows,state):
    set_services(rows,state['vocab']); admitted=[r for r in rows if r['host'] in state['medians']]
    rejected=collections.Counter(r['host'] for r in rows if r['host'] not in state['medians'])
    full=np.stack([np.r_[r['x'],r['x']-state['medians'][r['host']]] for r in admitted]) if admitted else np.empty((0,64))
    z=scale_apply(full,state['scaler']); finite=np.isfinite(z).all(axis=1)
    rows=[r for r,ok in zip(admitted,finite,strict=True) if ok]; z=z[finite]
    out={}; contributions=[]
    for block,bundle in state['models'].items():
        x=z[:,bundle['mask']]; results={}
        for key,model in bundle['models'].items():
            method=key.split('@')[0]; scores,parts=model_scores(method,model,x) if len(x) else (np.empty(0),np.empty_like(x))
            assert np.isfinite(scores).all()
            threshold=state['thresholds'][block][key]; leads=fold_leads(rows,scores,threshold)
            index={(r['host'],r['hour']):i for i,r in enumerate(rows)}; ranks={i:l['rank'] for l in leads for i in l['indices']}
            results[key]={'rows':rows,'index':index,'scores':scores,'parts':parts,'leads':leads,'ranks':ranks,'threshold':threshold,'flagged_hours':int((scores>threshold).sum()),'lead_count':len(leads)}
        out[block]=results
    dropped=[]
    for i in np.flatnonzero(~state['scaler']['keep']):
        dropped.append({'column':state['colnames'][i],'min':float(full[:,i].min()) if len(full) else None,'max':float(full[:,i].max()) if len(full) else None})
    return {'results':out,'no_history':dict(rejected),'non_finite':int((~finite).sum()),'dropped':dropped}


def hour_status(result,host,hour):
    i=result['index'].get((host,hour))
    if i is None: return {'status':'abstained','score':None,'rank':None,'flagged':False,'parts':None}
    flagged=bool(result['scores'][i]>result['threshold']); rank=result['ranks'].get(i)
    return {'status':('in lead rank '+str(rank)) if flagged else 'below T','score':float(result['scores'][i]),'rank':rank,'flagged':flagged,'parts':result['parts'][i]}


def attribute(clean,inj,host,labeled,state,case):
    table=[]; detected={}
    for block in inj['results']:
        detected[block]={}
        for method,r in inj['results'][block].items():
            cr=clean['results'][block][method]; hit=False
            for hour in sorted(labeled):
                a=hour_status(cr,host,hour); b=hour_status(r,host,hour)
                induced=b['flagged'] and (not a['flagged'] or b['score']>a['score'])
                qualifies=bool(induced and b['rank'] is not None and b['rank']<=10); hit|=qualifies
                changes=[]
                if b['parts'] is not None:
                    delta=b['parts']-(a['parts'] if a['parts'] is not None else 0)
                    names=np.array(state['colnames'])[state['models'][block]['mask']]
                    changes=[{'column':str(names[i]),'change':float(delta[i])} for i in np.argsort(-np.abs(delta))[:5]]
                table.append({'case':case,'block':block,'method':method,'host':state['mapping'][host],'hour':datetime.fromtimestamp(hour,timezone.utc).isoformat(),'clean_score':a['score'],'clean_status':a['status'],'clean_rank':a['rank'],'injected_score':b['score'],'injected_status':b['status'],'injected_rank':b['rank'],'induced':bool(induced),'qualifies':qualifies,'contribution_changes':changes})
            detected[block][method]=bool(hit)
    return detected,table


def freeze_state(state,work,extra):
    work=Path(work); work.mkdir(parents=True,exist_ok=True)
    payload=work/'frozen-state.pkl'
    with payload.open('wb') as f: pickle.dump(state,f,protocol=5)
    safe_history=[{**h,'host':state['mapping'][h['host']]} for h in state['history']]
    freeze={'schema_version':1,'frozen_at':instant(),'parameters':PARAMETERS,'versions':{n:importlib.metadata.version(n) for n in ('scikit-learn','numpy','pandas')},'python':platform.python_version(),'state_sha256':sha(payload),'home_net_sha256':hashlib.sha256(canonical(state['networks'])).hexdigest(),'implementation_sha256':{'notebook':sha(ROOT/'notebooks/conn_hosthour.ipynb'),'runner':sha(extra['runner_path']) if extra.get('runner_path') else None},'home_net_description':extra['home_net_description'],'fallback_used':extra['fallback_used'],'vocabulary':state['vocab'],'ranking':state['ranking'],'trimmed_hours':state['trimmed'],'scorability':safe_history,'scaler_sha256':hashlib.sha256(state['scaler']['center'].tobytes()+state['scaler']['divisor'].tobytes()+state['scaler']['keep'].tobytes()).hexdigest(),'surviving_columns':[n for n,k in zip(state['colnames'],state['scaler']['keep'],strict=True) if k],'thresholds':state['thresholds'],'targets':{k:state['mapping'][v] for k,v in state['targets'].items()},'donor':state['mapping'].get(state['donor']),'features':32,'columns_before_constants':64}
    json_write(work/'freeze.json',freeze)
    return freeze

def reload_state(work):
    work=Path(work); freeze=json.loads((work/'freeze.json').read_text()); assert sha(work/'frozen-state.pkl')==freeze['state_sha256']
    with (work/'frozen-state.pkl').open('rb') as f: state=pickle.load(f)
    assert state['thresholds']==freeze['thresholds'] and PARAMETERS==freeze['parameters']
    return state,freeze


def safe_coverage(coverage,state):
    rows=[]
    for r in coverage:
        r=dict(r)
        if 'host' in r: r['host']=state['mapping'].get(r['host'],'host-unseen-'+hashlib.sha256(r['host'].encode()).hexdigest()[:8])
        rows.append(r)
    return rows


def public_fixture(work):
    """35 dates, six hosts, >100 admitted hours/host; no private discovery."""
    work=Path(work); root=work/'synthetic'; root.mkdir(parents=True,exist_ok=True); rng=np.random.default_rng(SEEDS[0]); dates=REFERENCE+WEEKS[0]
    for d in dates:
        directory=root/str(d); directory.mkdir(exist_ok=True)
        with (directory/'conn.log').open('w') as f:
            base=datetime.combine(d,datetime.min.time(),timezone.utc).timestamp()
            for host in range(1,7):
                for hour in range(24):
                    for j in range(int(rng.integers(6,14))+host):
                        record={'_path':'conn','ts':base+hour*3600+float(rng.uniform(0,3599)),'id.orig_h':f'192.168.50.{host}','id.resp_h':f'198.51.100.{int(rng.integers(1,15))}','id.resp_p':int(rng.choice([53,80,443,8443])),'proto':str(rng.choice(['tcp','udp'])),'service':str(rng.choice(['dns','http','ssl',''])),'orig_bytes':int(rng.integers(0,10000))*(host+1),'resp_bytes':int(rng.integers(0,30000)),'duration':float(rng.uniform(0,30)),'conn_state':str(rng.choice(['SF','S1','S0','REJ'])),'local_orig':True}
                        f.write(json.dumps(record)+'\n')
    return root


class StudyReader:
    def __init__(self,root,work,real=False): self.root=Path(root); self.work=Path(work); self.real=real; self.events=[]; self.hashes=[]; self.times=[]
    def day(self,d):
        d=guard_day(d)
        if d not in REFERENCE: assert (self.work/'freeze.json').is_file(),'evaluation_before_freeze'
        paths=sorted((self.root/str(d)).glob('conn.*')); assert paths,'missing conn day'
        self.events.append({'dataset':'estate-'+str(d),'read_started':instant(),'evaluation':d not in REFERENCE})
        start=time.perf_counter()
        # Suppress operational details in public output; errors are handled by dataset ID.
        with contextlib.redirect_stdout(__import__('io').StringIO()),contextlib.redirect_stderr(__import__('io').StringIO()):
            frame=loader.load_logs(paths[0].parent,'conn',_files=paths,show_progress=False)
        for p in paths: self.hashes.append({'dataset':'estate-'+str(d),'sha256':sha(p),'bytes':p.stat().st_size})
        self.times.append({'phase':'reference_load' if d in REFERENCE else 'evaluation_load','dataset':str(d),'seconds':time.perf_counter()-start})
        return frame


def shifted(frame,target,week):
    assert not frame.empty,'empty selected capture'
    start=datetime.combine(week[1],datetime.min.time(),ZoneInfo('America/Chicago')).replace(hour=10).timestamp()
    # week[1] is Tuesday because weeks start Monday.
    df=frame.copy(); ts=pd.to_numeric(df['ts'],errors='coerce'); df=df.loc[np.isfinite(ts)].copy(); ts=ts.loc[df.index]
    delta=start-float(ts.min()); df['ts']=ts+delta; stop=datetime.combine(week[0]+timedelta(days=7),datetime.min.time(),timezone.utc).timestamp()
    clipped=int((df['ts']>=stop).sum()); df=df.loc[df['ts']<stop].copy(); df['src']=target; df['local_orig']=pd.NA
    return df,clipped


def synthetic_assertions():
    rows=[{'host':'h1','hour':h*3600} for h in (3,4,6,11)]+[{'host':'h2','hour':4*3600}]
    leads=fold_leads(rows,np.array([8.,9.1,8.,9.1,7.]),6.)
    assert len(leads)==3 and leads[0]['hours']==[10800,14400,21600] and leads[1]['hours']==[39600]
    x=np.array([[0.,1.,5.],[0.,2.,5.],[1.,3.,5.]])
    s=scale_fit(x); assert s['keep'].tolist()==[True,True,False] and np.isfinite(scale_apply(x,s)).all()
    rs=[{'x':np.zeros(32),'cats':{'none':8,'http':2},'n':10}]; set_services(rs,['http']); assert np.isclose(rs[0]['x'][13:23].sum(),1)
    return {'fold_worked_example':'passed','constant_scaler':'passed','service_none_partition':'passed'}


def verdict(detections,counts):
    ae='autoencoder'; unique=[r['case'] for r in detections if r[ae] and not r['robust-z'] and not r['PCA']]
    held=[i for i in unique if i>=3]
    wins=sum(r[ae] and (not r['robust-z'] or not r['PCA']) for r in detections)
    losses=sum(not r[ae] and (r['robust-z'] or r['PCA']) for r in detections)
    cost=all(row[ae]['flagged_hours']<=min(row['robust-z']['flagged_hours'],row['PCA']['flagged_hours']) and row[ae]['leads']<=min(row['robust-z']['leads'],row['PCA']['leads']) for row in counts)
    clauses={'a':bool(held),'b':wins-losses>=1,'c':cost}
    return {'outcome':'VIABLE' if all(clauses.values()) else 'NOT VIABLE','clauses':clauses,'heldout_unique_cases':held,'wins':wins,'losses':losses,'net_gain':wins-losses,'meaning':'incremental ranking of these injected cases; no general attack discrimination claim'}


def summarize_leads(evaluation,state,period):
    out=[]
    for block,methods in evaluation['results'].items():
        for method,r in methods.items():
            names=np.array(state['colnames'])[state['models'][block]['mask']]
            for lead in r['leads']:
                peak=max(lead['indices'],key=lambda i:r['scores'][i]); parts=r['parts'][peak]
                out.append({'period':period,'block':block,'method':method,'host':state['mapping'][lead['host']],'rank':lead['rank'],'score':lead['score'],'flagged_member_hours':[datetime.fromtimestamp(h,timezone.utc).isoformat() for h in lead['hours']],'top_contributions':[{'column':str(names[i]),'value':float(parts[i])} for i in np.argsort(-parts)[:5]]})
    return out


def score_bundle(bundle,state,freeze,coverage,reader):
    counts=[]; leads=[]; dropped=[]; cases=[]; hours=[]; controls=[]; variants=[]; nohist=[]
    evaluations={}
    for week_id,rows in bundle['clean'].items():
        e=evaluate(rows,state); evaluations[week_id]=e
        full=e['results']['full']; counts.append({m:{'flagged_hours':full[m]['flagged_hours'],'leads':full[m]['lead_count']} for m in METHODS})
        leads.extend(summarize_leads(e,state,'week-'+str(week_id)+'-clean'))
        dropped.extend([{'period':'week-'+str(week_id),**r} for r in e['dropped']]); nohist.append({'week':week_id,'hours':sum(e['no_history'].values()),'non_finite':e['non_finite'],'by_host':{state['mapping'].get(h,'host-unseen-'+hashlib.sha256(h.encode()).hexdigest()[:8]):n for h,n in e['no_history'].items()}})
    for case in bundle['cases']:
        e=evaluate(case['rows'],state); clean=evaluations[case['week']]
        det,table=attribute(clean,e,case['target'],case['labeled'],state,case['case']); hours.extend(table)
        cases.append({'case':case['case'],'scenario':case['scenario'],'placement':case['placement'],**{m:det['full'][m] for m in METHODS}})
        for block,methods in det.items():
            for m,yes in methods.items(): variants.append({'case':case['case'],'block':block,'method':m,'detected':yes})
        leads.extend(summarize_leads(e,state,'case-'+str(case['case'])))
    for control in bundle['controls']:
        e=evaluate(control['rows'],state); det,table=attribute(evaluations[control['week']],e,control['target'],control['labeled'],state,control['id'])
        controls.append({'control':control['id'],'week':control['week'],'host':state['mapping'][control['target']],'injected_rows':control['injected_rows'],'span_seconds':control['span_seconds'],'detected':{m:det['full'][m] for m in METHODS},'exploratory':True}); hours.extend(table)
        leads.extend(summarize_leads(e,state,control['id']))
    decision=verdict(cases,counts) if len(cases)==6 else {'outcome':'PUBLIC WITNESS','cases':len(cases),'meaning':'synthetic execution only; no efficacy claim'}
    return {'decision':decision,'freeze':freeze,'freeze_sha256':sha(reader.work/'freeze.json'),'counts':counts,'detections':cases,'per_hour':hours,'variants':variants,'controls':controls,'capture_summary':bundle['capture_summary'],'leads':leads,'dropped_axes':dropped,'coverage':safe_coverage(coverage,state),'scorability':freeze['scorability'],'no_history':nohist,'fits':state['fits'],'read_events':reader.events,'inputs':reader.hashes,'phase_times':reader.times,'guard':{'live_reserve_attempts':0,'allowed_date_ids_only':True},'limits':['presumed-benign estate','sensitivity placements are not independent compromises','constant axes can miss changes','strengthened existing hour may qualify','benign controls exploratory']}


def build_bundle(reader,state,captures,coverage,week_count=2,donor_frame=None):
    # Only the tiny selected capture subsets persist; no whole-week raw frame is retained.
    bundle={'clean':{},'cases':[],'controls':[],'capture_summary':[]}; active=[]
    for i,(scenario,cap,week_id) in enumerate(captures):
        for j,(placement,target) in enumerate(state['targets'].items()):
            frame,clipped=shifted(cap,target,WEEKS[week_id-1]); labeled=set((frame['ts'].to_numpy()//3600).astype('int64')*3600)
            case={'case':i*2+j+1,'scenario':scenario,'placement':placement,'target':target,'week':week_id,'rows':[],'labeled':labeled,'splice':frame}
            active.append(case)
            bundle['capture_summary'].append({'case':case['case'],'scenario':scenario,'rows':len(frame),'span_seconds':float(frame['ts'].max()-frame['ts'].min()),'clipped_rows':clipped,'internal_share':float(frame['dst'].map(lambda d:_ip_in_home_net(d,state['networks'])).mean())})
    controls=[]
    if donor_frame is not None and not donor_frame.empty:
        for week_id in range(1,week_count+1):
            for placement,target in state['targets'].items():
                frame,clipped=shifted(donor_frame,target,WEEKS[week_id-1]); controls.append({'id':'control-'+str(week_id)+'-'+placement,'week':week_id,'target':target,'rows':[],'labeled':set((frame['ts'].to_numpy()//3600).astype('int64')*3600),'splice':frame,'injected_rows':len(frame),'span_seconds':float(frame['ts'].max()-frame['ts'].min())})
    for week_id in range(1,week_count+1):
        clean=[]
        for d in WEEKS[week_id-1]:
            frame=reader.day(d); dayrows=aggregate(frame,state['networks'],coverage,'clean-'+str(week_id)); clean.extend(dayrows)
            start=datetime.combine(d,datetime.min.time(),timezone.utc).timestamp(); end=start+86400
            for copy in [c for c in active+controls if c['week']==week_id]:
                splice=copy['splice']; addition=splice.loc[(splice['ts']>=start)&(splice['ts']<end)]
                if addition.empty: copy['rows'].extend(dayrows); continue
                target=copy['target']; original=frame.loc[frame['src']==target]
                changed=pd.concat([original,addition],ignore_index=True)
                changedrows=aggregate(changed,state['networks'],coverage,'case-'+str(copy.get('case',copy.get('id'))))
                copy['rows'].extend([r for r in dayrows if r['host']!=target]+changedrows)
            del frame; gc.collect()
        bundle['clean'][week_id]=clean
    for copy in active+controls: del copy['splice']
    bundle['cases']=active; bundle['controls']=controls
    return bundle


def run_public(root):
    root=Path(root); root.mkdir(parents=True,exist_ok=True); check=parity_checks(); check.update(synthetic_assertions())
    fixture=public_fixture(root); coverage=[]; reader=StudyReader(fixture,root)
    reference=[]
    for d in REFERENCE: reference.extend(aggregate(reader.day(d),['192.168.0.0/16'],coverage,'reference'))
    state=training(reference,['192.168.0.0/16']); freeze_state(state,root,{'home_net_description':'synthetic private-range sources; documentation-range destinations','fallback_used':False}); state,freeze=reload_state(root)
    with tempfile.TemporaryDirectory(prefix='det2-demo-') as tmp:
        subprocess.run([sys.executable,str(ROOT/'demo/gen_corpus.py'),'--scenario','bench',tmp],check=True,stdout=subprocess.DEVNULL)
        path=Path(tmp)/'zeek/conn.log'; cap=loader.load_logs(path.parent,'conn',_files=[path],show_progress=False)
        smoke=[]; agg=aggregate(cap,['192.168.0.0/16'],smoke,'demo-smoke'); check['demo_smoke']={'loaded_rows':len(cap),'admitted_hours':len(agg)}
        bundle=build_bundle(reader,state,[('synthetic-bench-beacon',cap,1)],coverage,week_count=1)
    results=score_bundle(bundle,state,freeze,coverage,reader); results['checks']=check
    replay_state,_=reload_state(root); replay=score_bundle(bundle,replay_state,freeze,coverage,reader); assert canonical(results['detections'])==canonical(replay['detections']); results['replay_detection_bytes_equal']=True
    json_write(root/'results.json',results)
    return results


def prepare_manifest(work):
    """Metadata/lineage only; never open a capture or evaluation row here."""
    from sigwood.common import config as config_module
    work=Path(work); work.mkdir(parents=True,exist_ok=True)
    assert not (work/'freeze.json').exists(),'existing freeze; use replay, never overwrite'
    config=config_module.load(); raw=config_module.resolve_home_net(config); networks=raw or list(_DEFAULT_HOME_NET)
    collection=Path.home()/'.sigwood/bench/external/c2-beacon-pcaps'; notes=(collection/'NOTES.md').read_text()
    sources={}
    for label,needle,relative,ports in [('sliver','zeek/tunneled-c2-beaconing/','zeek/tunneled-c2-beaconing/conn.log',None),('numinon','zeek/numinon/','zeek/numinon/conn.log',[6767,7676])]:
        line=next(line for line in notes.splitlines() if needle in line and '|' in line)
        ips=re.findall(r'\b(?:\d{1,3}\.){3}\d{1,3}\b',line)
        assert len(ips)>=2,'capture target missing from lineage'
        path=collection/relative; assert path.is_file(),'capture missing'
        sources[label]={'path':str(path),'source':ips[0],'destination':ips[1],'ports':ports,'size_bytes':path.stat().st_size}
    protocol=Path.home()/'.sigwood/bench/protocol'
    prior=[]
    for scenario in ('phase1-6767','phase3-7676'):
        path=protocol/('leg-c-heldback-positive-numinon-'+scenario+'-frozen-v1.json'); d=json.loads(path.read_text()); assert scenario in d['scenario']
        prior.append({'id':d['scenario'],'sha256':sha(path),'target_rows':d['target_rows']})
    manifest={'created_at':instant(),'root':str(Path.home()/'.sigwood/exports/zeek'),'home_net':networks,'fallback_used':not bool(raw),'captures':sources,'lineage':{'notes_sha256':sha(collection/'NOTES.md'),'prior_frozen_results':prior},'allowed_dates':sorted(str(d) for d in ALLOWED)}
    json_write(work/'manifest.json',manifest); return manifest


def load_capture(spec,reader,label):
    assert (reader.work/'freeze.json').exists(),'capture_before_freeze'
    path=Path(spec['path']); start=time.perf_counter()
    with contextlib.redirect_stdout(__import__('io').StringIO()),contextlib.redirect_stderr(__import__('io').StringIO()): frame=loader.load_logs(path.parent,'conn',_files=[path],show_progress=False)
    reader.events.append({'dataset':label,'read_started':instant(),'evaluation':True,'capture_load':True})
    reader.hashes.append({'dataset':label,'sha256':sha(path),'bytes':path.stat().st_size}); reader.times.append({'phase':'capture_load','dataset':label,'seconds':time.perf_counter()-start})
    subset=frame.loc[frame['src']==spec['source']].copy(); assert len(subset),'capture target has no rows'
    # Establish target correspondence without exposing private identities.
    assert (subset['dst']==spec['destination']).any(),'capture destination lineage mismatch'
    return subset


def run_real(work):
    work=Path(work); parity_checks(); synthetic_assertions(); manifest=prepare_manifest(work); networks=manifest['home_net']
    reader=StudyReader(manifest['root'],work,True); coverage=[]; reference=[]; start=time.perf_counter()
    for d in REFERENCE:
        reference.extend(aggregate(reader.day(d),networks,coverage,'reference')); gc.collect()
    print(json.dumps({'phase':'reference_aggregated','hours':len(reference),'elapsed_seconds':time.perf_counter()-start}),flush=True)
    fit_start=time.perf_counter(); state=training(reference,networks); reader.times.append({'phase':'reference_aggregate_and_fit','seconds':time.perf_counter()-start}); reader.times.append({'phase':'fit','seconds':time.perf_counter()-fit_start})
    description='operator configuration, exact values retained only in external manifest' if not manifest['fallback_used'] else 'built-in effective private-network fallback; values retained externally'
    freeze_state(state,work,{'home_net_description':description,'fallback_used':manifest['fallback_used'],'runner_path':globals().get('EXECUTION_RUNNER')}); state,freeze=reload_state(work)
    freeze_mtime=(work/'freeze.json').stat().st_mtime_ns
    print(json.dumps({'phase':'frozen','freeze_sha256':sha(work/'freeze.json'),'scorable_hosts':len(state['medians']),'columns':int(state['scaler']['keep'].sum())}),flush=True)
    sliver=load_capture(manifest['captures']['sliver'],reader,'development-sliver')
    numinon=load_capture(manifest['captures']['numinon'],reader,'heldout-numinon')
    captures=[('sliver-ligolo',sliver,1)]
    for name,port in [('numinon-A',6767),('numinon-B',7676)]:
        subset=numinon.loc[numinon['port']==port].copy(); assert len(subset),'scenario selector empty'
        captures.append((name,subset,2))
    del numinon
    donor_frame=None
    if state['donor'] is not None:
        frame=reader.day(date(2026,7,7)); donor_frame=frame.loc[frame['src']==state['donor']].copy(); del frame
    bundle=build_bundle(reader,state,captures,coverage,donor_frame=donor_frame)
    assert (work/'freeze.json').stat().st_mtime_ns==freeze_mtime,'freeze modified'
    assert all(e['read_started']>freeze['frozen_at'] for e in reader.events if e['evaluation'])
    # Replays read compact cached observations, never the held-out source again.
    read_data={'events':reader.events,'hashes':reader.hashes,'times':reader.times}
    with (work/'scoring-bundle.pkl').open('wb') as f: pickle.dump((bundle,coverage,read_data),f,protocol=5)
    scoring=time.perf_counter(); results=score_bundle(bundle,state,freeze,coverage,reader); results['phase_times'].append({'phase':'scoring','seconds':time.perf_counter()-scoring})
    results['manifest_sha256']=sha(work/'manifest.json'); results['lineage']=manifest['lineage']; results['freeze_mtime_unchanged']=True
    json_write(work/'results.json',results); return results




## Public execution witness

The seeded fixture exercises the same loader, two-block reference fit, freeze, thresholds, leads and paired attribution. It also runs the demo aggregation smoke and independent contract examples. Synthetic detections are execution evidence only.

In [3]:
import tempfile
_public_tmp = tempfile.TemporaryDirectory(prefix="det2-public-")
public_result = run_public(Path(_public_tmp.name))
print(json.dumps({"checks": public_result["checks"], "decision": public_result["decision"], "detections": public_result["detections"], "replay": public_result["replay_detection_bytes_equal"]}, indent=2))

{
  "checks": {
    "address_parity": "passed",
    "reserve_guard_self_test": "passed; no file discovery/open",
    "features": 32,
    "matrix_columns_before_constants": 64,
    "fold_worked_example": "passed",
    "constant_scaler": "passed",
    "service_none_partition": "passed",
    "demo_smoke": {
      "loaded_rows": 530,
      "admitted_hours": 24
    }
  },
  "decision": {
    "outcome": "PUBLIC WITNESS",
    "cases": 2,
    "meaning": "synthetic execution only; no efficacy claim"
  },
  "detections": [
    {
      "case": 1,
      "scenario": "synthetic-bench-beacon",
      "placement": "quiet",
      "robust-z": true,
      "PCA": true,
      "autoencoder": true
    },
    {
      "case": 2,
      "scenario": "synthetic-bench-beacon",
      "placement": "busy",
      "robust-z": true,
      "PCA": true,
      "autoencoder": true
    }
  ],
  "replay": true
}
